## Analysis extension #6 

This notebook is wired to the **AE1b** and reuses the exact clean developmental/Burdziak matrices, audited MLM embeddings, AE5 developmental split, AE3 developmental-state definitions, and AE3 biological-sample weighting.

AE6 keeps two questions separate.

### Primary question - representation dependence

Hold the **joint development + Burdziak representation-learning logic** fixed, and change the representation:

1. audited clean **joint MLM** (the AE1b primary seed);
2. conventional **joint log-normalized/HVG expression PCA**;
3. **joint scVI** (two seeds).

In all three spaces, the downstream developmental reference is fitted using **developmental reference cells only**. Cancer can participate in learning the common representation, but it does not define the developmental PCA subspace against which cells are evaluated.

This directly addresses the question: *are the geometric conclusions peculiar to the transformer representation?*

### Secondary control - tumor participation during representation learning

The AE1b **development-only MLM** seeds are kept as a separate leave-tumor-out control. They ask whether the principal directional/residual conclusions disappear when cancer does not participate in representation learning.

### Audited conventions carried forward

- Developmental reference / held-out split is reproduced exactly from AE5: stratified by developmental identity × day with seed 13.
- Revised developmental states/directions are the final AE3 definitions.
- Burdziak ductal / transitional / malignant-ductal centroids are **equal-biological-sample weighted** using `batch`, as in final AE3.
- MLM embeddings are loaded from AE1b's definitive cached `X_dev_cls.npy` / `X_cancer_cls.npy` with exact obs-name order assertions.
- Tabula Muris adult pancreas is reconstructed from the original FACS release and used as an external query.
- Absolute projection fractions are **not** interpreted as directly comparable across representation geometries. The matched-developmental-variance section is the main cross-representation sensitivity analysis.

(*note that in the notebook last running AE was labeled WP, work_package*)

In [28]:
from pathlib import Path
import os, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad
from scipy import sparse
from sklearn.decomposition import PCA, IncrementalPCA

from lightning.pytorch.callbacks import Callback

import torch


RANDOM_SEED = 13
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
# ============================================================
# AUDITED CONFIGURATION — AE1b is the single source of truth
# ============================================================

COMMONPATH = Path("/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data")
AUDITED_ROOT = COMMONPATH / "revision_audited_mlm"
MANIFEST_PATH = AUDITED_ROOT / "manifest.json"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"{MANIFEST_PATH}\n"
        "Run the definitive AE1b audited retraining notebook first."
    )

with open(MANIFEST_PATH) as f:
    AUDIT_MANIFEST = json.load(f)

DEV_H5AD = Path(AUDIT_MANIFEST["dev_h5ad"])
CANCER_H5AD = Path(AUDIT_MANIFEST["cancer_h5ad"])

PRIMARY_SEED = int(AUDIT_MANIFEST["primary_seed"])
ALL_SEEDS = [int(s) for s in AUDIT_MANIFEST["seeds"]]
DEV_ONLY_SEEDS = ALL_SEEDS

JOINT_MODEL_DIR = Path(
    AUDIT_MANIFEST["models"]["joint"][str(PRIMARY_SEED)]
)
DEV_ONLY_MODEL_DIRS = {
    int(seed): Path(
        AUDIT_MANIFEST["models"]["development_only"][str(seed)]
    )
    for seed in DEV_ONLY_SEEDS
}

TM_DIR = COMMONPATH / "adult_normal" / "tabula_muris"
TM_PANCREAS_COUNTS = TM_DIR / "FACS" / "Pancreas-counts.csv"
TM_FACS_ANNOT = TM_DIR / "annotations_facs.csv"
TM_FACS_META = TM_DIR / "metadata_FACS.csv"

AE5_OUT = AUDITED_ROOT / "analysis" / "ae5_adult_normal_calibration"
AE5_CELL_METRICS = AE5_OUT / "ae5_cell_metrics.csv.gz"

OUTDIR = (
    AUDITED_ROOT
    / "analysis"
    / "ap6_representation_dependence_joint_primary"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

DEV_REFERENCE_FRACTION = 0.80
K_DEV = 75

# Joint conventional expression PCA
N_HVG = 3000
PCA_REP_DIM = 256
PCA_BATCH_SIZE = 4096

# Joint scVI
SCVI_LATENT_DIM = 256
SCVI_SEEDS = [17, 42]
SCVI_MAX_EPOCHS = 150

# Development-only PCA/scVI are not needed for the primary reviewer test.
RUN_DEV_ONLY_ALT_REPS = False

print("Audited root:        ", AUDITED_ROOT)
print("AP1b manifest:       ", MANIFEST_PATH)
print("Primary joint seed:  ", PRIMARY_SEED)
print("Development matrix:  ", DEV_H5AD)
print("Burdziak matrix:     ", CANCER_H5AD)
print("Primary joint MLM:   ", JOINT_MODEL_DIR)
print("AP5 calibration:     ", AE5_CELL_METRICS)
print("AP6 output directory:", OUTDIR)


Audited root:         /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm
WP1b manifest:        /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/manifest.json
Primary joint seed:   17
Development matrix:   /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/development/A_dev_aligned_clean.h5ad
Burdziak matrix:      /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/pdac/A_cancer_aligned_clean.h5ad
Primary joint MLM:    /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/joint/seed_17/model
WP5 calibration:      /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp5_adult_normal_calibration/wp5_cell_metrics.csv.gz
WP6 output directory: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation_dependence_joint_primary


## 1. Load the aligned developmental and Burdziak count matrices


In [30]:
A_dev = sc.read_h5ad(DEV_H5AD)
A_can = sc.read_h5ad(CANCER_H5AD)

assert A_dev.n_obs == int(AUDIT_MANIFEST["n_dev_cells"])
assert A_can.n_obs == int(AUDIT_MANIFEST["n_cancer_cells"])
assert A_dev.n_vars == int(AUDIT_MANIFEST["n_genes"])
assert A_can.n_vars == int(AUDIT_MANIFEST["n_genes"])

assert A_dev.n_vars == A_can.n_vars
assert np.array_equal(
    A_dev.var_names.astype(str),
    A_can.var_names.astype(str),
)
assert "condition" in A_can.obs.columns
assert "cancer_state" in A_can.obs.columns
assert "batch" in A_can.obs.columns


def assert_countlike_matrix(name, X, max_probe=250_000, tol=1e-4):
    """
    scVI with a negative-binomial likelihood requires raw/count-like input.
    Check non-negativity and integer-likeness on a deterministic probe.
    """
    if sparse.issparse(X):
        data = np.asarray(X.data)
    else:
        data = np.asarray(X).ravel()

    if data.size == 0:
        raise ValueError(f"{name}: empty expression matrix")

    if not np.all(np.isfinite(data)):
        raise ValueError(f"{name}: non-finite values found")

    if float(data.min()) < 0:
        raise ValueError(f"{name}: negative values found; raw counts expected")

    step = max(1, data.size // max_probe)
    probe = data[::step][:max_probe]
    max_integer_deviation = float(
        np.max(np.abs(probe - np.rint(probe)))
    )

    if max_integer_deviation > tol:
        raise ValueError(
            f"{name}: values are not count-like "
            f"(max |x-round(x)|={max_integer_deviation:.3g}). "
            "Stop before scVI training."
        )

    print(
        f"{name}: count-like preflight PASS "
        f"(probe n={len(probe):,}, max integer deviation="
        f"{max_integer_deviation:.3g})"
    )


assert_countlike_matrix("A_dev.X", A_dev.X)
assert_countlike_matrix("A_can.X", A_can.X)

print("Development:", A_dev.shape)
print("Burdziak:   ", A_can.shape)
print(A_can.obs["condition"].value_counts().sort_index())


A_dev.X: count-like preflight PASS (probe n=250,000, max integer deviation=0)
A_can.X: count-like preflight PASS (probe n=250,000, max integer deviation=0)
Development: (34760, 15376)
Burdziak:    (28131, 15376)
condition
K1      4503
K1.5    2476
K2      2079
K3      3338
K4      1254
K5      5364
K6      3491
N1      3225
N2      2401
Name: count, dtype: int64


## 2. Reconstruct the exact adult-normal Tabula Muris query set used in AE5

This reproduces the authoritative original Tabula Muris FACS/Smart-seq2 pancreas release checks used in final AE5:

- 1,961 raw pancreas cells;
- 1,564 released annotated/QC-passing pancreas cells;
- 1,395 adult epithelial cells used for calibration:
  - 182 acinar,
  - 161 ductal,
  - 1,052 endocrine.

Adult cells do **not** participate in fitting expression PCA or scVI. They are transformed/encoded only after the corresponding joint representation is frozen.


In [ ]:
EXPECTED_RAW_PANCREAS_CELLS = 1961
EXPECTED_RELEASED_PANCREAS_CELLS = 1564
EXPECTED_ADULT_EPITHELIAL_CELLS = 1395

ADULT_LINEAGE_MAP = {
    "pancreatic acinar cell": "acinar",
    "pancreatic ductal cell": "ductal",
    "type B pancreatic cell": "endocrine",
    "pancreatic A cell": "endocrine",
    "pancreatic D cell": "endocrine",
    "pancreatic PP cell": "endocrine",
}

EXPECTED_ADULT_COUNTS = {
    "acinar": 182,
    "ductal": 161,
    "endocrine": 1052,
}

TM_counts = pd.read_csv(
    TM_PANCREAS_COUNTS,
    index_col=0,
    low_memory=False,
)
TM_annot = pd.read_csv(
    TM_FACS_ANNOT,
    low_memory=False,
)
TM_meta = pd.read_csv(
    TM_FACS_META,
    low_memory=False,
)

required_annot_cols = {
    "cell",
    "cell_ontology_class",
    "mouse.id",
    "mouse.sex",
    "plate.barcode",
    "tissue",
}
missing_cols = required_annot_cols - set(TM_annot.columns)
if missing_cols:
    raise KeyError(
        "annotations_facs.csv is missing required columns: "
        + ", ".join(sorted(missing_cols))
    )

ann_pan = TM_annot[
    TM_annot["tissue"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("pancreas")
].copy()
ann_pan["cell"] = ann_pan["cell"].astype(str)

if ann_pan["cell"].duplicated().any():
    raise ValueError(
        "Duplicate pancreas cell IDs in annotations_facs.csv"
    )

TM_counts.index = TM_counts.index.astype(str)
TM_counts.columns = TM_counts.columns.astype(str)

raw_cells = pd.Index(TM_counts.columns)
annot_cells = pd.Index(ann_pan["cell"])
raw_set = set(raw_cells)
annot_set = set(annot_cells)

assert len(raw_set) == EXPECTED_RAW_PANCREAS_CELLS
assert len(annot_set) == EXPECTED_RELEASED_PANCREAS_CELLS
assert len(annot_set - raw_set) == 0

is_ercc = TM_counts.index.str.startswith("ERCC-")
TM_counts_no_ercc = TM_counts.loc[~is_ercc].copy()

tm_n_counts = TM_counts_no_ercc.sum(axis=0)
tm_n_genes = (TM_counts_no_ercc > 0).sum(axis=0)
tm_qc_pass = (tm_n_genes >= 500) & (tm_n_counts > 50_000)
qc_pass_cells = set(tm_qc_pass.index[tm_qc_pass])

assert qc_pass_cells == annot_set, (
    "Reconstructed Tabula Muris FACS QC does not exactly match "
    "the released pancreas annotations."
)

# Preserve raw count-matrix order, exactly as in final AP5.
keep_cells = [c for c in raw_cells if c in annot_set]

A_tm = ad.AnnData(
    X=sparse.csr_matrix(
        TM_counts_no_ercc.loc[:, keep_cells]
        .T
        .to_numpy(dtype=np.float32, copy=False)
    ),
    obs=ann_pan.set_index("cell").loc[keep_cells].copy(),
    var=pd.DataFrame(
        index=TM_counts_no_ercc.index.astype(str)
    ),
)
A_tm.var["symbol"] = A_tm.var_names.astype(str)
A_tm.var_names_make_unique()

A_tm.obs["n_counts_tm"] = tm_n_counts.loc[keep_cells].to_numpy()
A_tm.obs["n_genes_tm"] = tm_n_genes.loc[keep_cells].to_numpy()
A_tm.obs["adult_lineage"] = (
    A_tm.obs["cell_ontology_class"].map(ADULT_LINEAGE_MAP)
)

A_adult_source = A_tm[
    A_tm.obs["adult_lineage"].notna()
].copy()

assert A_adult_source.n_obs == EXPECTED_ADULT_EPITHELIAL_CELLS

lineage_counts = (
    A_adult_source.obs["adult_lineage"]
    .value_counts()
)
for lineage, expected in EXPECTED_ADULT_COUNTS.items():
    found = int(lineage_counts.get(lineage, 0))
    assert found == expected, (lineage, found, expected)

# Every retained adult lineage should be represented in every mouse.
mouse_lineage = pd.crosstab(
    A_adult_source.obs["mouse.id"],
    A_adult_source.obs["adult_lineage"],
)
if (mouse_lineage == 0).any().any():
    raise ValueError(
        "At least one adult lineage is absent from at least one mouse."
    )

assert_countlike_matrix(
    "A_adult_source.X",
    A_adult_source.X,
)

print("Authoritative TM pancreas:", A_tm.shape)
print("Adult epithelial query:   ", A_adult_source.shape)
print(lineage_counts.to_string())
print("\nMouse x adult lineage:")
print(mouse_lineage.to_string())

# Dense CSV tables are no longer needed after AnnData construction.
del TM_counts, TM_counts_no_ercc
gc.collect()


A_adult_source.X: count-like preflight PASS (probe n=250,000, max integer deviation=0)
Authoritative TM pancreas: (1564, 23341)
Adult epithelial query:    (1395, 23341)
adult_lineage
endocrine    1052
acinar        182
ductal        161

Mouse x adult lineage:
adult_lineage  acinar  ductal  endocrine
mouse.id                                
3_10_M             36      58        244
3_38_F             64      27        270
3_39_F             31      29        253
3_8_M              51      47        285


94

In [ ]:
def strip_ens_version(x):
    return str(x).split(".")[0]


def as_csr(X):
    if sparse.issparse(X):
        return X.tocsr().astype(np.float32)
    return sparse.csr_matrix(
        np.asarray(X, dtype=np.float32)
    )


def get_symbol_series(adata):
    for col in [
        "symbol",
        "gene_symbol",
        "gene_symbols",
        "gene_name",
        "feature_name",
    ]:
        if col in adata.var.columns:
            return pd.Series(
                adata.var[col].astype(str).values,
                index=adata.var_names.astype(str),
            )
    return pd.Series(
        adata.var_names.astype(str),
        index=adata.var_names.astype(str),
    )


def align_counts_to_audited_reference(
    A_source,
    A_reference,
):
    """
    Map source raw counts into the exact audited A_dev/A_can gene order.

    Direct Ensembl matches are used first; otherwise source symbols are
    matched to the symbols attached to the audited developmental matrix.
    Multiple source features mapping to one audited gene are summed.
    """
    model_genes = list(
        map(str, A_reference.var_names)
    )
    model_index = {
        strip_ens_version(g): j
        for j, g in enumerate(model_genes)
    }
    exact_model = {
        strip_ens_version(g): g
        for g in model_genes
    }

    ref_symbols = get_symbol_series(A_reference)
    symbol_to_model = {}

    for ens, sym in zip(
        A_reference.var_names.astype(str),
        ref_symbols.astype(str),
    ):
        key = str(sym).upper()
        # First mapping retained, matching the audited AE5 logic.
        symbol_to_model.setdefault(key, str(ens))

    source_names = list(
        map(str, A_source.var_names)
    )
    source_symbols = (
        get_symbol_series(A_source)
        .astype(str)
        .values
    )

    src_idx = []
    dst_idx = []
    direct = 0
    by_symbol = 0

    for i, (g, sym) in enumerate(
        zip(source_names, source_symbols)
    ):
        g0 = strip_ens_version(g)
        mg = None

        if g0 in exact_model:
            mg = exact_model[g0]
            direct += 1
        else:
            mg = symbol_to_model.get(
                str(sym).upper()
            )
            if mg is None:
                mg = symbol_to_model.get(
                    str(g).upper()
                )
            if mg is not None:
                by_symbol += 1

        if mg is not None:
            src_idx.append(i)
            dst_idx.append(
                model_index[
                    strip_ens_version(mg)
                ]
            )

    if not src_idx:
        raise ValueError(
            "No adult genes mapped to the audited reference."
        )

    M = sparse.coo_matrix(
        (
            np.ones(
                len(src_idx),
                dtype=np.float32,
            ),
            (
                np.asarray(src_idx),
                np.asarray(dst_idx),
            ),
        ),
        shape=(
            A_source.n_vars,
            A_reference.n_vars,
        ),
    ).tocsr()

    Xmap = (
        as_csr(A_source.X)
        @ M
    ).tocsr()

    mapped_model_genes = len(set(dst_idx))

    print(
        f"Mapped {len(src_idx):,}/{A_source.n_vars:,} "
        f"source features onto "
        f"{mapped_model_genes:,}/{A_reference.n_vars:,} "
        f"audited genes "
        f"(direct={direct:,}, symbol={by_symbol:,})."
    )

    return ad.AnnData(
        X=Xmap,
        obs=A_source.obs.copy(),
        var=A_reference.var.copy(),
    )


A_adult = align_counts_to_audited_reference(A_adult_source,A_dev)
A_adult.var_names = A_dev.var_names.copy()

assert A_adult.n_obs == 1395
assert A_adult.n_vars == A_dev.n_vars
assert np.array_equal(
    A_adult.var_names.astype(str),
    A_dev.var_names.astype(str),
)
assert_countlike_matrix("A_adult.X", A_adult.X)

print("Adult aligned to audited reference:", A_adult.shape)


Mapped 13,280/23,341 source features onto 13,280/15,376 audited genes (direct=0, symbol=13,280).
A_adult.X: count-like preflight PASS (probe n=250,000, max integer deviation=0)
Adult aligned to audited reference: (1395, 15376)


## 3. Reproduce the exact AE5 developmental reference / held-out split

Final AE5 used one shared stratified split based on `dev_state_union × day`, with `RANDOM_SEED = 13` and 80% of each stratum assigned to the developmental reference.

The code below intentionally mirrors the final AE5 implementation, including the `groupby(strata.values)` iteration, so that the same cell identities — not merely the same fraction — are used.


In [ ]:
def choose_dev_label_col(obs):
    for c in [
        "dev_state_union",
        "celltype_update",
        "endo_category",
        "dev_tier",
        "cell_type",
        "celltype",
    ]:
        if c in obs.columns:
            return c
    raise KeyError(
        "No developmental identity column found."
    )


def choose_stage_col(obs):
    for c in [
        "day",
        "stage",
        "developmental_stage",
        "embryonic_day",
        "timepoint",
    ]:
        if c in obs.columns:
            return c
    return None


def choose_ae3_day_col(obs):
    for c in [
        "dev_day",
        "day",
        "stage",
        "developmental_stage",
        "embryonic_day",
        "timepoint",
    ]:
        if c in obs.columns:
            return c
    raise KeyError(
        "No developmental day column found for final AE3 directions."
    )


DEV_LABEL_COL = choose_dev_label_col(A_dev.obs)
DEV_SPLIT_DAY_COL = choose_stage_col(A_dev.obs)
DEV_DAY_COL = choose_ae3_day_col(A_dev.obs)

print("Developmental identity:", DEV_LABEL_COL)
print("Developmental stage:   ", DEV_SPLIT_DAY_COL)
print("AE3 day column:         ", DEV_DAY_COL)

# IMPORTANT: fresh RNG, no previous draws. This exactly mirrors final AE5.
split_rng = np.random.default_rng(RANDOM_SEED)

strata = A_dev.obs[DEV_LABEL_COL].astype(str)
if DEV_SPLIT_DAY_COL is not None:
    strata = (
        strata
        + "|"
        + A_dev.obs[DEV_SPLIT_DAY_COL].astype(str)
    )

ref_idx = []
hold_idx = []

for _, pos in pd.Series(
    np.arange(A_dev.n_obs),
    index=A_dev.obs_names,
).groupby(strata.values):

    ids = pos.to_numpy()
    split_rng.shuffle(ids)

    if len(ids) == 1:
        ref_idx.extend(ids)
        continue

    n_ref = max(
        1,
        min(
            len(ids) - 1,
            int(
                np.floor(
                    DEV_REFERENCE_FRACTION
                    * len(ids)
                )
            ),
        ),
    )

    ref_idx.extend(ids[:n_ref])
    hold_idx.extend(ids[n_ref:])

ref_idx = np.array(
    sorted(ref_idx),
    dtype=int,
)
hold_idx = np.array(
    sorted(hold_idx),
    dtype=int,
)

assert len(
    np.intersect1d(
        ref_idx,
        hold_idx,
    )
) == 0

# Definitive AE5 split counts.
assert len(ref_idx) == 27678, (
    f"Expected AE5 reference n=27678, found {len(ref_idx)}"
)
assert len(hold_idx) == 7082, (
    f"Expected AE5 held-out n=7082, found {len(hold_idx)}"
)

print("Reference developmental cells:", len(ref_idx))
print("Held-out developmental cells: ", len(hold_idx))


Developmental identity: dev_state_union
Developmental stage:    day
WP3 day column:          dev_day
Reference developmental cells: 27678
Held-out developmental cells:  7082


## 4. Shared geometric calculations

The developmental state definitions are the final AE3 definitions. Developmental centroids are evaluated both cell-weighted and equal-day-weighted, but the principal directional comparison uses **equal-day developmental centroids**.

For cancer, the final AE3 convention is stricter: each Burdziak biological sample (`batch`) contributes one sample mean to the ductal, transitional, or malignant-ductal state centroid. Thus large samples do not dominate malignant displacement vectors.


In [ ]:
def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    den = np.linalg.norm(a) * np.linalg.norm(b)
    return np.nan if den == 0 else float(np.dot(a, b) / den)


def fit_dev_geometry(Z_dev, ref_idx, k=75):
    Z_dev = np.asarray(Z_dev, dtype=np.float32)
    mu = Z_dev[ref_idx].mean(axis=0)
    Xref = Z_dev[ref_idx] - mu

    k_eff = min(
        int(k),
        Z_dev.shape[1] - 1,
        len(ref_idx) - 1,
    )

    pca = PCA(
        n_components=k_eff,
        svd_solver="full",
    ).fit(Xref)

    return {
        "mu": mu,
        "basis": pca.components_.T,
        "k": k_eff,
        "dev_variance": float(
            pca.explained_variance_ratio_.sum()
        ),
    }


def project_metrics(Z, geom):
    X = np.asarray(Z, dtype=np.float32) - geom["mu"]
    B = geom["basis"]
    S = X @ B
    P = S @ B.T
    R = X - P

    denom = np.einsum("ij,ij->i", X, X)
    numer = np.einsum("ij,ij->i", P, P)

    q = np.divide(
        numer,
        denom,
        out=np.full_like(
            numer,
            np.nan,
            dtype=np.float32,
        ),
        where=denom > 0,
    )
    q = np.clip(q, 0.0, 1.0)

    return {
        "projection_fraction": q,
        "residual_norm": np.linalg.norm(R, axis=1),
        "residual_fraction": np.sqrt(
            np.clip(1.0 - q, 0.0, 1.0)
        ),
        "scores": S,
        "residuals": R,
    }


# ------------------------------------------------------------
# Final AE3 developmental-state definitions
# ------------------------------------------------------------

def revised_ae3_masks(dev_obs):
    ct = dev_obs["celltype_update"].astype(str)
    day = pd.to_numeric(
        dev_obs[DEV_DAY_COL],
        errors="coerce",
    )

    return {
        "MPC_like": (
            (ct == "Pancreatic acinar cells")
            & (day <= 11.5)
        ).to_numpy(),

        "tip_acinar": (
            (ct == "Pancreatic acinar cells")
            & (day >= 13.75)
            & (day <= 16.5)
        ).to_numpy(),

        "endocrine_secondary": (
            (ct == "Pancreatic islets")
            & (day >= 13.75)
            & (day <= 16.5)
        ).to_numpy(),

        "endocrine_late": (
            (ct == "Pancreatic islets")
            & (day >= 16.75)
        ).to_numpy(),

        "biliary_early": (
            (ct == "Biliary epithelial cells")
            & (day >= 11.75)
            & (day <= 13.5)
        ).to_numpy(),

        "biliary_secondary": (
            (ct == "Biliary epithelial cells")
            & (day >= 13.75)
            & (day <= 16.5)
        ).to_numpy(),
    }


AE3_STATE_MASKS = revised_ae3_masks(A_dev.obs)

EXPECTED_AE3_STATE_COUNTS = {
    "MPC_like": 164,
    "tip_acinar": 2871,
    "endocrine_secondary": 633,
    "endocrine_late": 556,
    "biliary_early": 126,
    "biliary_secondary": 601,
}

observed_ae3_counts = {
    name: int(mask.sum())
    for name, mask in AE3_STATE_MASKS.items()
}

assert observed_ae3_counts == EXPECTED_AE3_STATE_COUNTS, (
    "Final AE3 developmental-state counts do not match the "
    f"audited specification: {observed_ae3_counts}"
)

print(
    "AE3 developmental-state counts:",
    observed_ae3_counts,
)


def equal_day_centroid(scores, dev_obs, mask, min_day_n=3):
    days = pd.to_numeric(
        dev_obs[DEV_DAY_COL],
        errors="coerce",
    ).to_numpy()

    centroids = []

    for d in np.sort(
        pd.unique(
            days[
                mask
                & np.isfinite(days)
            ]
        )
    ):
        m = mask & np.isclose(days, d)

        if int(m.sum()) >= min_day_n:
            centroids.append(
                np.asarray(
                    scores[m].mean(axis=0)
                ).ravel()
            )

    if not centroids:
        raise ValueError(
            "No developmental days passed the equal-day minimum."
        )

    return np.vstack(centroids).mean(axis=0)


def build_ae3_vectors(dev_scores, ref_idx, weighting="cell"):
    ref_mask = np.zeros(A_dev.n_obs, dtype=bool)
    ref_mask[ref_idx] = True

    C = {}

    for name, state_mask in AE3_STATE_MASKS.items():
        m = state_mask & ref_mask

        if weighting == "cell":
            C[name] = np.asarray(
                dev_scores[m].mean(axis=0)
            ).ravel()
        elif weighting == "equal_day":
            C[name] = equal_day_centroid(
                dev_scores,
                A_dev.obs,
                m,
            )
        else:
            raise ValueError(
                f"Unknown developmental weighting: {weighting}"
            )

    return {
        "MPC_to_tip_acinar":
            C["tip_acinar"] - C["MPC_like"],

        "MPC_to_endocrine_islet":
            C["endocrine_secondary"] - C["MPC_like"],

        "endocrine_maturation":
            C["endocrine_late"] - C["endocrine_secondary"],

        "biliary_maturation":
            C["biliary_secondary"] - C["biliary_early"],
    }


# ------------------------------------------------------------
# Final AE3 malignant-state convention:
# equal biological-sample weight within each state
# ------------------------------------------------------------

SAMPLE_COL = "batch"
CANCER_STATE_ORDER = [
    "ductal",
    "transitional",
    "malignant_ductal",
]

_can_obs_for_centroids = A_can.obs.copy()
_can_obs_for_centroids["_row_position"] = np.arange(
    A_can.n_obs
)


def _sample_groups_for_state(obs, state):
    sub = obs.loc[
        obs["cancer_state"].astype(str).eq(state)
    ]

    return sub.groupby(
        SAMPLE_COL,
        observed=True,
        sort=False,
    )


def sample_equal_state_centroids(X, obs):
    """
    One mean per biological sample, then equal average of
    sample means within each cancer state.
    """
    X = np.asarray(X)
    C = {}

    for state in CANCER_STATE_ORDER:
        sample_means = []

        for sample, group in _sample_groups_for_state(
            obs,
            state,
        ):
            pos = group[
                "_row_position"
            ].to_numpy(dtype=int)

            if pos.size == 0:
                continue

            sample_mean = X[pos].mean(axis=0)

            if not np.all(np.isfinite(sample_mean)):
                raise ValueError(
                    "Non-finite sample centroid for "
                    f"state={state!r}, sample={sample!r}"
                )

            sample_means.append(sample_mean)

        if not sample_means:
            raise ValueError(
                "No observed biological samples found "
                f"for cancer state {state!r}"
            )

        C[state] = np.mean(
            np.stack(sample_means),
            axis=0,
        )

    return C


sample_counts_by_state = {
    state: int(
        A_can.obs.loc[
            A_can.obs["cancer_state"].astype(str).eq(state),
            SAMPLE_COL,
        ]
        .astype(str)
        .nunique()
    )
    for state in CANCER_STATE_ORDER
}

EXPECTED_SAMPLE_COUNTS = {
    "ductal": 4,
    "transitional": 11,
    "malignant_ductal": 6,
}

assert sample_counts_by_state == EXPECTED_SAMPLE_COUNTS, (
    "Burdziak biological-sample counts no longer match final AE3: "
    f"{sample_counts_by_state}"
)

print(
    "Burdziak biological samples per state:",
    sample_counts_by_state,
)


def malignant_direction_cosines(dev_scores, can_scores, ref_idx):
    C_can = sample_equal_state_centroids(
        can_scores,
        _can_obs_for_centroids,
    )

    malignant_vectors = {
        "early":
            C_can["transitional"] - C_can["ductal"],

        "late":
            C_can["malignant_ductal"] - C_can["transitional"],

        "total":
            C_can["malignant_ductal"] - C_can["ductal"],
    }

    rows = []

    for weighting in ["cell", "equal_day"]:
        dvecs = build_ae3_vectors(
            dev_scores,
            ref_idx,
            weighting,
        )

        for transition, mvec in malignant_vectors.items():
            for dname, dvec in dvecs.items():
                c = cosine(mvec, dvec)

                rows.append({
                    "centroid_weighting":
                        weighting,
                    "cancer_centroid_weighting":
                        "equal_biological_sample",
                    "transition":
                        transition,
                    "developmental_direction":
                        dname,
                    "cosine":
                        c,
                    "abs_cosine":
                        abs(c),
                })

    return pd.DataFrame(rows)


def eta_squared(values, labels):
    x = np.asarray(values, dtype=float)
    lab = np.asarray(labels).astype(str)
    mu = np.nanmean(x)
    ss_total = np.nansum((x - mu) ** 2)

    if ss_total <= 0:
        return np.nan

    ss_between = 0.0

    for g in pd.unique(lab):
        m = lab == g
        gm = np.nanmean(x[m])
        ss_between += m.sum() * (gm - mu) ** 2

    return float(ss_between / ss_total)


def multivariate_between_fraction(X, labels):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels).astype(str)
    mu = X.mean(axis=0)
    ss_total = np.sum((X - mu) ** 2)

    if ss_total <= 0:
        return np.nan

    ss_between = 0.0

    for g in pd.unique(labels):
        m = labels == g
        gm = X[m].mean(axis=0)
        ss_between += (
            m.sum()
            * np.sum((gm - mu) ** 2)
        )

    return float(ss_between / ss_total)


def residual_structure_summary(residuals):
    cond = A_can.obs["condition"].astype(str).to_numpy()
    state = A_can.obs["cancer_state"].astype(str).to_numpy()

    k_mask = np.isin(
        cond,
        [
            "K1",
            "K1.5",
            "K2",
            "K3",
            "K4",
            "K5",
            "K6",
        ],
    )

    R = np.asarray(
        residuals[k_mask],
        dtype=np.float32,
    )
    cond_k = cond[k_mask]
    state_k = state[k_mask]

    ncomp = min(
        30,
        R.shape[1],
        R.shape[0] - 1,
    )

    rpca = PCA(
        n_components=ncomp,
        svd_solver="randomized",
        random_state=RANDOM_SEED,
    )
    scores = rpca.fit_transform(R)

    eta_condition = [
        eta_squared(
            scores[:, j],
            cond_k,
        )
        for j in range(min(10, ncomp))
    ]

    eta_state = [
        eta_squared(
            scores[:, j],
            state_k,
        )
        for j in range(min(10, ncomp))
    ]

    return {
        "n_K_cells":
            int(k_mask.sum()),

        "residual_pc_variance_first10":
            float(
                rpca.explained_variance_ratio_[
                    :min(10, ncomp)
                ].sum()
            ),

        "residual_pc_variance_first30":
            float(
                rpca.explained_variance_ratio_.sum()
            ),

        "between_condition_fraction_raw_residual":
            multivariate_between_fraction(
                R,
                cond_k,
            ),

        "between_broad_state_fraction_raw_residual":
            multivariate_between_fraction(
                R,
                state_k,
            ),

        "max_eta2_condition_first10":
            float(np.nanmax(eta_condition)),

        "max_eta2_broad_state_first10":
            float(np.nanmax(eta_state)),

        "mean_eta2_condition_first10":
            float(np.nanmean(eta_condition)),

        "mean_eta2_broad_state_first10":
            float(np.nanmean(eta_state)),
    }


def population_projection_summary(
    dev_hold_metrics,
    can_metrics,
    adult_metrics=None,
):
    rows = []

    def add(name, vals):
        vals = np.asarray(vals, dtype=float)
        vals = vals[np.isfinite(vals)]

        if len(vals) == 0:
            raise ValueError(
                f"No finite projection values for {name}"
            )

        rows.append({
            "population":
                name,
            "n_cells":
                len(vals),
            "projection_median":
                float(np.median(vals)),
            "projection_q25":
                float(np.quantile(vals, 0.25)),
            "projection_q75":
                float(np.quantile(vals, 0.75)),
        })

    held_labels = (
        A_dev.obs
        .iloc[hold_idx][DEV_LABEL_COL]
        .astype(str)
        .to_numpy()
    )

    add(
        "heldout_dev_ductal",
        dev_hold_metrics[
            "projection_fraction"
        ][held_labels == "ductal"],
    )

    cond = A_can.obs["condition"].astype(str).to_numpy()
    qc = can_metrics["projection_fraction"]

    for c in [
        "N1",
        "N2",
        "K1",
        "K1.5",
        "K2",
        "K3",
        "K4",
        "K5",
        "K6",
    ]:
        add(
            c,
            qc[cond == c],
        )

    add(
        "K_series_all",
        qc[
            np.isin(
                cond,
                [
                    "K1",
                    "K1.5",
                    "K2",
                    "K3",
                    "K4",
                    "K5",
                    "K6",
                ],
            )
        ],
    )

    if adult_metrics is not None:
        lin = (
            A_adult.obs["adult_lineage"]
            .astype(str)
            .to_numpy()
        )

        add(
            "adult_ductal",
            adult_metrics[
                "projection_fraction"
            ][lin == "ductal"],
        )

    return pd.DataFrame(rows)


def evaluate_representation(
    name,
    Z_dev,
    Z_can,
    Z_adult=None,
    k=K_DEV,
):
    Z_dev = np.asarray(
        Z_dev,
        dtype=np.float32,
    )
    Z_can = np.asarray(
        Z_can,
        dtype=np.float32,
    )

    if Z_dev.shape[0] != A_dev.n_obs:
        raise ValueError(
            f"{name}: developmental row count mismatch"
        )
    if Z_can.shape[0] != A_can.n_obs:
        raise ValueError(
            f"{name}: cancer row count mismatch"
        )

    geom = fit_dev_geometry(Z_dev,ref_idx,k)

    dev_hold = project_metrics(Z_dev[hold_idx],geom)
    dev_all = project_metrics(Z_dev,geom)
    
    can = project_metrics(Z_can,geom)

    adult = (
        project_metrics(
            Z_adult,
            geom,
        )
        if Z_adult is not None
        else None
    )

    return {
        "projection_summary":
            population_projection_summary(
                dev_hold,
                can,
                adult,
            ),

        "direction_cosines":
            malignant_direction_cosines(
                dev_all["scores"],
                can["scores"],
                ref_idx,
            ),

        "residual_summary":
            pd.DataFrame([
                residual_structure_summary(
                    can["residuals"]
                )
            ]),

        "cancer_cell_metrics":
            pd.DataFrame({
                "cell_id":
                    A_can.obs_names.astype(str),
                "projection_fraction":
                    can["projection_fraction"],
                "residual_fraction":
                    can["residual_fraction"],
            }),
    }


WP3 developmental-state counts: {'MPC_like': 164, 'tip_acinar': 2871, 'endocrine_secondary': 633, 'endocrine_late': 556, 'biliary_early': 126, 'biliary_secondary': 601}
Burdziak biological samples per state: {'ductal': 4, 'transitional': 11, 'malignant_ductal': 6}


## 5. Joint conventional log-normalized/HVG expression PCA

This is the conventional-expression analogue of the **audited clean joint MLM**.

- HVGs are selected from the combined clean development + Burdziak corpus.
- PCA loadings are fitted to that same combined corpus.
- Adult normal is transformed only after the representation is frozen.
- The developmental reference/subspace is then fitted using the exact AE5 developmental reference cells only.

Cancer therefore participates in learning the common representation, but not in defining the developmental reference geometry.


In [35]:
# ------------------------------------------------------------
# 5A. Joint HVG selection
# ------------------------------------------------------------

X_joint_counts = sparse.vstack([A_dev.X, A_can.X],format="csr").astype(np.float32)
A_joint_hvg = ad.AnnData(X=X_joint_counts,var=A_dev.var.copy())  # (n_dev + n_can) × n_genes.

sc.pp.normalize_total(A_joint_hvg, target_sum=1e4)
sc.pp.log1p(A_joint_hvg)
sc.pp.highly_variable_genes(A_joint_hvg,n_top_genes=N_HVG,flavor="seurat",subset=False)

# variability in the combined population rather than variability only in the development population.
hvg_idx = np.where(A_joint_hvg.var["highly_variable"].to_numpy())[0]
hvg_names = A_dev.var_names[hvg_idx].astype(str).to_numpy()

pd.DataFrame({"gene": hvg_names}).to_csv(OUTDIR / "joint_expression_pca_hvgs.csv",index=False)

print("Joint HVGs:", len(hvg_idx))

del A_joint_hvg
gc.collect()


def lognorm_selected(X, selected_idx, target_sum=1e4):
    if sparse.issparse(X):
        X = X.tocsr().astype(np.float32)
    else:
        X = sparse.csr_matrix(np.asarray(X, dtype=np.float32))

    lib = np.asarray(X.sum(axis=1)).ravel()
    scale = np.zeros(len(lib), dtype=np.float32)
    nz = lib > 0
    scale[nz] = float(target_sum) / lib[nz]

    Y = (
        sparse.diags(scale)
        @ X[:, selected_idx].tocsr().astype(np.float32)
    ).tocsr()
    Y.data = np.log1p(Y.data)
    return Y


Xdev_log = lognorm_selected(A_dev.X, hvg_idx)
Xcan_log = lognorm_selected(A_can.X, hvg_idx)
Xadult_log = lognorm_selected(A_adult.X, hvg_idx)

Xjoint_log = sparse.vstack([Xdev_log, Xcan_log],format="csr")

# Ordinary PCA often requires converting the entire matrix to dense format, 
# which can consume a lot of memory. IncrementalPCA processes the matrix in blocks.
def fit_incremental_pca(X, n_components, batch_size):
    n = X.shape[0]
    bounds = list(range(0, n, batch_size)) + [n]

    # IncrementalPCA requires every partial-fit block to contain at least
    # n_components rows. Merge a tiny last block into the previous one.
    if (
        len(bounds) >= 3
        and (bounds[-1] - bounds[-2]) < n_components
    ):
        bounds.pop(-2)

    # Create and fit the model
    model = IncrementalPCA(n_components=n_components,batch_size=batch_size)

    for i in range(len(bounds) - 1):
        a, b = bounds[i], bounds[i + 1]
        block = X[a:b].toarray().astype(np.float32)
        model.partial_fit(block)
        print(f"  PCA fit rows {a:,}:{b:,}")

    return model


def transform_incremental_pca(model, X, batch_size):
    chunks = []
    for a in range(0, X.shape[0], batch_size):
        b = min(a + batch_size, X.shape[0])
        block = X[a:b].toarray().astype(np.float32)
        chunks.append(
            model.transform(block).astype(np.float32)
        )
    return np.vstack(chunks)


joint_expr_pca = fit_incremental_pca(Xjoint_log,n_components=PCA_REP_DIM,batch_size=PCA_BATCH_SIZE)
Zjoint_expr = transform_incremental_pca(joint_expr_pca,Xjoint_log,PCA_BATCH_SIZE)

Zdev_expr = Zjoint_expr[:A_dev.n_obs]
Zcan_expr = Zjoint_expr[A_dev.n_obs:]
Zadult_expr = transform_incremental_pca(joint_expr_pca,Xadult_log,PCA_BATCH_SIZE)

np.savez_compressed(
    OUTDIR / "joint_expression_pca_model.npz",
    mean_=joint_expr_pca.mean_,
    components_=joint_expr_pca.components_,
    explained_variance_ratio_=joint_expr_pca.explained_variance_ratio_,
    hvg_names=hvg_names,
)

results = {}

results["joint_expression_PCA"] = evaluate_representation(
    "joint_expression_PCA",
    Zdev_expr,
    Zcan_expr,
    Zadult_expr,
    K_DEV,
)


Joint HVGs: 3000
  PCA fit rows 0:4,096
  PCA fit rows 4,096:8,192
  PCA fit rows 8,192:12,288
  PCA fit rows 12,288:16,384
  PCA fit rows 16,384:20,480
  PCA fit rows 20,480:24,576
  PCA fit rows 24,576:28,672
  PCA fit rows 28,672:32,768
  PCA fit rows 32,768:36,864
  PCA fit rows 36,864:40,960
  PCA fit rows 40,960:45,056
  PCA fit rows 45,056:49,152
  PCA fit rows 49,152:53,248
  PCA fit rows 53,248:57,344
  PCA fit rows 57,344:61,440
  PCA fit rows 61,440:62,891


## 6. Joint scVI

The scVI control is learned from the same **clean combined development + Burdziak corpus** used for the primary joint-representation comparison.

The model is deliberately **not** given development-versus-Burdziak as a `batch_key`: explicitly correcting that dataset label could remove genuine biological differences together with study effects.

The matrices are checked above to be non-negative and count-like before the negative-binomial scVI likelihood is used.

Adult Tabula Muris cells are not used for training or adaptation. After training, the fixed encoder assigns them posterior-mean latent coordinates as an external query.

If `import scvi` fails, install `scvi-tools` in the notebook environment, restart the kernel, and rerun.


In [ ]:
try:
    import scvi
    print("scvi-tools:", scvi.__version__)
except ImportError as e:
    raise ImportError(
        "scvi-tools is required for AE6 joint scVI. "
        "Run `%pip install scvi-tools`, restart the kernel, and rerun."
    ) from e

print('dl_num_workers:', scvi.settings.dl_num_workers)

scvi-tools: 1.4.2
dl_num_workers: 0


In [ ]:
def make_scvi_adata(
    X,
    obs,
    var_names,
):
    if sparse.issparse(X):
        Xout = X.tocsr().astype(np.float32)
    else:
        Xout = sparse.csr_matrix(
            np.asarray(X, dtype=np.float32)
        )

    return ad.AnnData(
        X=Xout,
        obs=obs.copy(),
        var=pd.DataFrame(
            index=pd.Index(
                var_names.astype(str)
            )
        ),
    )


joint_obs = pd.concat(
    [
        A_dev.obs.copy(),
        A_can.obs.copy(),
    ],
    axis=0,
    sort=False,
)

joint_obs["ae6_source_dataset"] = (
    ["development"] * A_dev.n_obs
    + ["burdziak"] * A_can.n_obs
)

joint_obs.index = (
    [
        f"DEV::{x}"
        for x in A_dev.obs_names.astype(str)
    ]
    + [
        f"CAN::{x}"
        for x in A_can.obs_names.astype(str)
    ]
)

scvi_joint = make_scvi_adata(
    sparse.vstack(
        [
            A_dev.X[:, hvg_idx],
            A_can.X[:, hvg_idx],
        ],
        format="csr",
    ),
    joint_obs,
    hvg_names,
)

scvi_adult = make_scvi_adata(
    A_adult.X[:, hvg_idx],
    A_adult.obs,
    hvg_names,
)

# Preflight checks before expensive training.
assert scvi_joint.n_vars == scvi_adult.n_vars
assert np.array_equal(
    scvi_joint.var_names.astype(str),
    scvi_adult.var_names.astype(str),
)
assert scvi_joint.n_obs == A_dev.n_obs + A_can.n_obs
assert scvi_adult.n_obs == A_adult.n_obs

print(
    "scVI inputs OK:",
    "joint =",
    scvi_joint.shape,
    "adult =",
    scvi_adult.shape,
)


def encode_frozen_scvi_query(
    reference_model,
    query_adata,
):
    """
    Encode adult normal with the already-trained joint scVI encoder.

    No query training and no scArches adaptation are performed.
    `give_mean=True` returns the posterior-mean latent coordinates.
    """
    q = query_adata.copy()

    ref_vars = np.asarray(
        reference_model.adata
        .var_names
        .astype(str)
    )
    query_vars = np.asarray(
        q.var_names.astype(str)
    )

    if not np.array_equal(
        ref_vars,
        query_vars,
    ):
        raise ValueError(
            "Query genes are not identical and in the same order "
            "as the trained scVI model."
        )

    try:
        Z = reference_model.get_latent_representation(
            adata=q,
            give_mean=True,
        )
    except Exception as e:
        raise RuntimeError(
            "Frozen external scVI query encoding failed in this "
            "scvi-tools version. Stop here rather than adapting/retraining "
            "the adult query, because AE6 requires adult normal to remain "
            "out of training."
        ) from e

    return Z


class EpochProgress(Callback):
    def on_train_epoch_end(
        self,
        trainer,
        pl_module,
    ):
        epoch = trainer.current_epoch + 1

        if epoch == 1 or epoch % 10 == 0:
            metrics = trainer.callback_metrics

            elbo = metrics.get("elbo_train")
            recon = metrics.get("reconstruction_loss_train")
            kl = metrics.get("kl_local_train")

            parts = [
                f"Epoch {epoch:3d}"
            ]

            if elbo is not None:
                parts.append(
                    f"ELBO={float(elbo):.4f}"
                )

            if recon is not None:
                parts.append(
                    f"recon={float(recon):.4f}"
                )

            if kl is not None:
                parts.append(
                    f"KL={float(kl):.4f}"
                )

            print(
                " | ".join(parts)
            )


def train_joint_scvi(seed):
    scvi.settings.seed = int(seed)

    joint = scvi_joint.copy()

    # Deliberately no batch_key:
    # development vs Burdziak is not treated as a nuisance variable.
    scvi.model.SCVI.setup_anndata(
        joint
    )

    model = scvi.model.SCVI(
        joint,
        n_latent=SCVI_LATENT_DIM,
        n_hidden=256,
        n_layers=2,
        gene_likelihood="nb",
        dispersion="gene",
    )

    model.train(
        max_epochs=SCVI_MAX_EPOCHS,
        early_stopping=True,
        accelerator="auto",
        devices="auto",
        enable_progress_bar=False,
        callbacks=[
            EpochProgress()
        ],
    )

    model_dir = (
        OUTDIR
        / f"joint_scVI_seed{seed}_model"
    )
    model.save(
        model_dir,
        overwrite=True,
    )

    # Training-corpus latent representation: posterior means.
    Zjoint = (
        model.get_latent_representation(
            give_mean=True
        )
        .astype(np.float32)
    )

    # Truly external adult query: encode only, no training/adaptation.
    Zad = (
        encode_frozen_scvi_query(
            model,
            scvi_adult,
        )
        .astype(np.float32)
    )

    Zdev = Zjoint[:A_dev.n_obs]
    Zcan = Zjoint[A_dev.n_obs:]

    assert Zdev.shape == (
        A_dev.n_obs,
        SCVI_LATENT_DIM,
    )
    assert Zcan.shape == (
        A_can.n_obs,
        SCVI_LATENT_DIM,
    )
    assert Zad.shape == (
        A_adult.n_obs,
        SCVI_LATENT_DIM,
    )

    return (
        Zdev,
        Zcan,
        Zad,
    )


for seed in SCVI_SEEDS:
    print(
        "\nTraining joint scVI seed",
        seed,
    )

    (
        Zdev_scvi,
        Zcan_scvi,
        Zad_scvi,
    ) = train_joint_scvi(seed)

    tag = f"joint_scVI_seed{seed}"

    results[tag] = evaluate_representation(
        tag,
        Zdev_scvi,
        Zcan_scvi,
        Zad_scvi,
        K_DEV,
    )

    del (
        Zdev_scvi,
        Zcan_scvi,
        Zad_scvi,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()


Seed set to 17


scVI inputs OK: joint = (62891, 3000) adult = (1395, 3000)

Training joint scVI seed 17


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jfpoyatos/miniconda3/envs/plasticity/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=79` in the `DataLoader` to improve performance.
/home/jfpoyatos/miniconda3/envs/plasticity/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=79` in the `DataLoader` to imp

Epoch   1 | ELBO=1284.7969 | recon=988.8082 | KL=295.9888
Epoch  10 | ELBO=991.0874 | recon=775.1511 | KL=215.9363
Epoch  20 | ELBO=889.3076 | recon=762.1496 | KL=127.1580
Epoch  30 | ELBO=853.3746 | recon=756.6982 | KL=96.6764
Epoch  40 | ELBO=836.2292 | recon=753.4904 | KL=82.7388
Epoch  50 | ELBO=827.1213 | recon=751.5988 | KL=75.5225
Epoch  60 | ELBO=820.6122 | recon=750.3760 | KL=70.2362
Epoch  70 | ELBO=815.6721 | recon=749.6636 | KL=66.0086
Epoch  80 | ELBO=811.8768 | recon=748.9963 | KL=62.8805
Epoch  90 | ELBO=808.9003 | recon=748.7361 | KL=60.1642
Epoch 100 | ELBO=806.5654 | recon=748.7326 | KL=57.8327
Epoch 110 | ELBO=804.3970 | recon=748.7512 | KL=55.6459
Epoch 120 | ELBO=802.1683 | recon=748.5998 | KL=53.5685
Epoch 130 | ELBO=800.5776 | recon=748.7676 | KL=51.8101
Epoch 140 | ELBO=799.1716 | recon=749.0132 | KL=50.1584


`Trainer.fit` stopped: `max_epochs=150` reached.


Epoch 150 | ELBO=797.6282 | recon=749.1677 | KL=48.4604
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


... (2 outputs left out)

/home/jfpoyatos/miniconda3/envs/plasticity/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=79` in the `DataLoader` to improve performance.
/home/jfpoyatos/miniconda3/envs/plasticity/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=79` in the `DataLoader` to improve performance.


Epoch   1 | ELBO=1281.4603 | recon=997.3993 | KL=284.0610
Epoch  10 | ELBO=994.2535 | recon=774.8600 | KL=219.3935
Epoch  20 | ELBO=892.6898 | recon=762.4862 | KL=130.2036
Epoch  30 | ELBO=854.5117 | recon=756.7714 | KL=97.7403
Epoch  40 | ELBO=837.8066 | recon=753.7484 | KL=84.0582
Epoch  50 | ELBO=830.5707 | recon=753.2433 | KL=77.3274
Epoch  60 | ELBO=822.0938 | recon=750.7198 | KL=71.3740
Epoch  70 | ELBO=817.5632 | recon=750.0896 | KL=67.4736
Epoch  80 | ELBO=813.8061 | recon=749.4484 | KL=64.3577
Epoch  90 | ELBO=810.9866 | recon=749.3321 | KL=61.6546
Epoch 100 | ELBO=808.5668 | recon=749.3265 | KL=59.2403
Epoch 110 | ELBO=806.3480 | recon=749.2065 | KL=57.1415
Epoch 120 | ELBO=804.1294 | recon=749.1112 | KL=55.0182
Epoch 130 | ELBO=802.4600 | recon=749.2886 | KL=53.1714
Epoch 140 | ELBO=800.6609 | recon=749.3342 | KL=51.3267


`Trainer.fit` stopped: `max_epochs=150` reached.


Epoch 150 | ELBO=799.2751 | recon=749.5576 | KL=49.7176
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


Training reminder: scVI optimizes the variational objective (ELBO), balancing count reconstruction against KL regularization of the latent posterior. For downstream AE6 geometry, `get_latent_representation(give_mean=True)` uses the posterior-mean coordinates.


## 7. Audited MLM reference representations

The primary transformer representation is the **AE1b joint MLM primary seed** (`PRIMARY_SEED` from `manifest.json`).

The two **AE1b development-only MLM** seeds remain the secondary leave-tumor-out training-composition control.

No MLM inference is recomputed here. AE6 consumes the definitive cached CLS embeddings written by AE1b and requires exact developmental and Burdziak obs-name order matches.


In [ ]:
MLM_DIRS = {
    "joint_MLM":
        JOINT_MODEL_DIR,

    **{
        f"development_only_MLM_seed{seed}":
            DEV_ONLY_MODEL_DIRS[int(seed)]
        for seed in DEV_ONLY_SEEDS
    },
}


def load_audited_mlm_embeddings(
    model_dir,
):
    """
    Load the definitive AE1b CLS embeddings and verify exact cell order.
    No fallback recomputation is allowed in the audited AE6.
    """
    model_dir = Path(model_dir)

    required = [
        model_dir / "X_dev_cls.npy",
        model_dir / "X_cancer_cls.npy",
        model_dir / "dev_obs_names.npy",
        model_dir / "cancer_obs_names.npy",
    ]

    missing = [
        str(p)
        for p in required
        if not p.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Missing definitive AE1b MLM cache(s):\n"
            + "\n".join(missing)
        )

    dev_names = np.load(
        model_dir
        / "dev_obs_names.npy",
        allow_pickle=True,
    ).astype(str)

    can_names = np.load(
        model_dir
        / "cancer_obs_names.npy",
        allow_pickle=True,
    ).astype(str)

    expected_dev = (
        A_dev.obs_names
        .astype(str)
        .to_numpy()
    )
    expected_can = (
        A_can.obs_names
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        dev_names,
        expected_dev,
    ):
        raise ValueError(
            f"{model_dir}: developmental obs-name/order mismatch"
        )

    if not np.array_equal(
        can_names,
        expected_can,
    ):
        raise ValueError(
            f"{model_dir}: Burdziak obs-name/order mismatch"
        )

    Zdev = np.load(
        model_dir
        / "X_dev_cls.npy"
    ).astype(np.float32)

    Zcan = np.load(
        model_dir
        / "X_cancer_cls.npy"
    ).astype(np.float32)

    if Zdev.shape[0] != A_dev.n_obs:
        raise ValueError(
            f"{model_dir}: developmental embedding row mismatch"
        )

    if Zcan.shape[0] != A_can.n_obs:
        raise ValueError(
            f"{model_dir}: cancer embedding row mismatch"
        )

    if not np.all(np.isfinite(Zdev)):
        raise ValueError(
            f"{model_dir}: non-finite developmental embedding values"
        )

    if not np.all(np.isfinite(Zcan)):
        raise ValueError(
            f"{model_dir}: non-finite cancer embedding values"
        )

    print(
        "Audited MLM cache:",
        model_dir,
        "| DEV",
        Zdev.shape,
        "| CANCER",
        Zcan.shape,
    )

    return Zdev, Zcan


for name, model_dir in MLM_DIRS.items():
    print(
        "\nLoading:",
        name,
    )

    (
        Zdev_mlm,
        Zcan_mlm,
    ) = load_audited_mlm_embeddings(
        model_dir
    )

    results[name] = evaluate_representation(
        name,
        Zdev_mlm,
        Zcan_mlm,
        None,
        K_DEV,
    )



Loading: joint_MLM
Audited MLM cache: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/joint/seed_17/model | DEV (34760, 256) | CANCER (28131, 256)

Loading: development_only_MLM_seed17
Audited MLM cache: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/development_only/seed_17/model | DEV (34760, 256) | CANCER (28131, 256)

Loading: development_only_MLM_seed42
Audited MLM cache: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/development_only/seed_42/model | DEV (34760, 256) | CANCER (28131, 256)


## 8. Import the exact final-AE5 adult-ductal MLM calibration

For expression PCA and scVI, adult cells are encoded directly as frozen external queries in this notebook.

For the audited MLMs, final AE5 already performed the adult query/tokenization audit. AE6 therefore imports the exact adult-ductal projection values from final AE5 rather than re-encoding adult cells a second time.

For the primary joint MLM, both `DEV` and `CANCER` domain-token assumptions are retained explicitly. The relevant AE5 encoder name is `joint_seed{PRIMARY_SEED}`.


In [ ]:
if not AE5_CELL_METRICS.exists():
    raise FileNotFoundError(
        f"Missing final AE5 table: {AE5_CELL_METRICS}"
    )

ae5_metrics = pd.read_csv(
    AE5_CELL_METRICS,
    low_memory=False,
)

ae5_metrics["projection_fraction"] = pd.to_numeric(
    ae5_metrics["projection_fraction"],
    errors="raise",
)

required_ae5_cols = [
    "encoder",
    "domain_assumption",
    "reference_group",
    "group",
    "projection_fraction",
]

missing_ae5_cols = [
    c
    for c in required_ae5_cols
    if c not in ae5_metrics.columns
]

if missing_ae5_cols:
    raise KeyError(
        "AE5 metrics table is missing columns: "
        f"{missing_ae5_cols}"
    )

print(
    ae5_metrics[
        required_ae5_cols
    ].dtypes
)


def ae5_adult_ductal_row(
    encoder,
    domain_assumption,
):
    m = (
        (
            ae5_metrics["encoder"]
            .astype(str)
            == encoder
        )
        & (
            ae5_metrics[
                "domain_assumption"
            ]
            .astype(str)
            == domain_assumption
        )
        & (
            ae5_metrics[
                "reference_group"
            ]
            .astype(str)
            == "adult_normal"
        )
        & (
            ae5_metrics["group"]
            .astype(str)
            == "adult_ductal"
        )
    )

    x = (
        ae5_metrics.loc[
            m,
            "projection_fraction",
        ]
        .to_numpy(dtype=float)
    )

    if len(x) != 161:
        raise ValueError(
            "Expected 161 adult ductal cells for "
            f"{encoder}/{domain_assumption}, "
            f"found {len(x)}"
        )

    if not np.all(np.isfinite(x)):
        raise ValueError(
            "Non-finite adult ductal projection values found for "
            f"{encoder}/{domain_assumption}"
        )

    return {
        "population":
            "adult_ductal",
        "n_cells":
            len(x),
        "projection_median":
            float(np.median(x)),
        "projection_q25":
            float(np.quantile(x, 0.25)),
        "projection_q75":
            float(np.quantile(x, 0.75)),
    }


# Add final-AE5 adult ductal to both development-only MLM results.
for seed in DEV_ONLY_SEEDS:
    rname = f"development_only_MLM_seed{seed}"

    row = ae5_adult_ductal_row(
        f"development_only_seed{seed}",
        "DEV",
    )

    results[
        rname
    ][
        "projection_summary"
    ] = pd.concat(
        [
            results[
                rname
            ][
                "projection_summary"
            ],
            pd.DataFrame([
                row
            ]),
        ],
        ignore_index=True,
    )


# Primary clean joint MLM adult-domain ambiguity:
# retain both final-AE5 variants explicitly.
joint_base = (
    results[
        "joint_MLM"
    ][
        "projection_summary"
    ].copy()
)

joint_projection_variants = {}

joint_ae5_encoder = (
    f"joint_seed{PRIMARY_SEED}"
)

for dom in [
    "DEV",
    "CANCER",
]:
    row = ae5_adult_ductal_row(
        joint_ae5_encoder,
        dom,
    )

    joint_projection_variants[
        f"joint_MLM_adult{dom}"
    ] = pd.concat(
        [
            joint_base,
            pd.DataFrame([
                row
            ]),
        ],
        ignore_index=True,
    )

print(
    "Final AE5 adult-ductal MLM calibration imported successfully."
)
print(
    "Primary joint AE5 encoder:",
    joint_ae5_encoder,
)


encoder                 object
domain_assumption       object
reference_group         object
group                   object
projection_fraction    float64
dtype: object
Final WP5 adult-ductal MLM calibration imported successfully.
Primary joint WP5 encoder: joint_seed17


## 9. Primary and secondary AE6 summaries

### Primary representation-dependence comparison

Compare the audited primary joint MLM, joint expression PCA, and two joint scVI seeds.

The fixed-`K=75` tables are retained for continuity, but because `K=75` captures very different fractions of developmental variance in the different representations, the **matched-developmental-variance section below is the principal cross-representation comparison**.

### Secondary training-composition control

Compare the primary joint MLM with the two AE1b development-only MLM seeds. This is conceptually separate from representation dependence.


In [ ]:
PRIMARY_NAMES = [
    "joint_expression_PCA",
    *[f"joint_scVI_seed{s}" for s in SCVI_SEEDS],
    "joint_MLM",
]

SECONDARY_NAMES = [
    "joint_MLM",
    *[
        f"development_only_MLM_seed{s}"
        for s in DEV_ONLY_SEEDS
    ],
]

ORDER_POPULATIONS = [
    "adult_ductal",
    "N1",
    "N2",
    "K1",
    "K1.5",
    "K2",
    "K3",
    "K4",
    "K5",
    "K6",
]

# ============================================================
# PRIMARY 1 — population projection ordering
# ============================================================

primary_projection_tables = []

for name in PRIMARY_NAMES:
    if name == "joint_MLM":
        continue

    t = results[name]["projection_summary"].copy()
    t["representation"] = name
    primary_projection_tables.append(t)

for name, t0 in joint_projection_variants.items():
    t = t0.copy()
    t["representation"] = name
    primary_projection_tables.append(t)

primary_projection = pd.concat(
    primary_projection_tables,
    ignore_index=True,
)

primary_projection.to_csv(
    OUTDIR / "ae6_primary_joint_projection_ordering.csv",
    index=False,
)

primary_wide = (
    primary_projection[
        primary_projection["population"].isin(ORDER_POPULATIONS)
    ]
    .pivot_table(
        index="population",
        columns="representation",
        values="projection_median",
        aggfunc="first",
    )
    .reindex(ORDER_POPULATIONS)
)

primary_rank_corr = primary_wide.corr(method="spearman")
primary_rank_corr.to_csv(
    OUTDIR / "ae6_primary_joint_population_rank_spearman.csv"
)

primary_rank_order = pd.DataFrame([
    {
        "representation": rep,
        "descending_projection_order":
            " > ".join(
                primary_wide[rep]
                .dropna()
                .sort_values(ascending=False)
                .index
                .tolist()
            ),
    }
    for rep in primary_wide.columns
])

primary_rank_order.to_csv(
    OUTDIR / "ae6_primary_joint_population_rank_order.csv",
    index=False,
)

# ============================================================
# PRIMARY 2 — per-cell Burdziak projection concordance
# ============================================================

primary_can = None

for name in PRIMARY_NAMES:
    d = (
        results[name]["cancer_cell_metrics"][
            ["cell_id", "projection_fraction"]
        ]
        .copy()
        .rename(columns={"projection_fraction": name})
    )

    primary_can = (
        d
        if primary_can is None
        else primary_can.merge(
            d,
            on="cell_id",
            how="inner",
            validate="one_to_one",
        )
    )

assert len(primary_can) == A_can.n_obs

primary_cell_corr = (
    primary_can
    .drop(columns="cell_id")
    .corr(method="spearman")
)

primary_cell_corr.to_csv(
    OUTDIR / "ae6_primary_joint_burdziak_cell_projection_spearman.csv"
)
primary_can.to_csv(
    OUTDIR / "ae6_primary_joint_burdziak_cell_projection_scores.csv.gz",
    index=False,
)

# ============================================================
# PRIMARY 3 — revised AE3 malignant-direction cosines
# ============================================================

primary_cosines = []

for name in PRIMARY_NAMES:
    t = results[name]["direction_cosines"].copy()
    t["representation"] = name
    primary_cosines.append(t)

primary_cosines = pd.concat(
    primary_cosines,
    ignore_index=True,
)

primary_cosines.to_csv(
    OUTDIR / "ae6_primary_joint_revised_ae3_direction_cosines.csv",
    index=False,
)

# ============================================================
# PRIMARY 4 — K-series-only residual structure
# ============================================================

primary_residual = []

for name in PRIMARY_NAMES:
    t = results[name]["residual_summary"].copy()
    t["representation"] = name
    primary_residual.append(t)

primary_residual = pd.concat(
    primary_residual,
    ignore_index=True,
)

primary_residual.to_csv(
    OUTDIR / "ae6_primary_joint_kseries_residual_structure.csv",
    index=False,
)

# ============================================================
# SECONDARY — joint MLM vs development-only MLM
# ============================================================

secondary_projection_tables = []

for name, t0 in joint_projection_variants.items():
    t = t0.copy()
    t["model"] = name
    secondary_projection_tables.append(t)

for name in SECONDARY_NAMES:
    if name == "joint_MLM":
        continue

    t = results[name]["projection_summary"].copy()
    t["model"] = name
    secondary_projection_tables.append(t)

secondary_projection = pd.concat(secondary_projection_tables,ignore_index=True)
secondary_projection.to_csv(OUTDIR / "ae6_secondary_joint_vs_devonly_MLM_projection.csv",index=False)

secondary_cosines = []
secondary_residual = []

for name in SECONDARY_NAMES:
    t = results[name]["direction_cosines"].copy()
    t["model"] = name
    secondary_cosines.append(t)

    r = results[name]["residual_summary"].copy()
    r["model"] = name
    secondary_residual.append(r)

pd.concat(
    secondary_cosines,
    ignore_index=True,
).to_csv(
    OUTDIR / "ae6_secondary_joint_vs_devonly_MLM_direction_cosines.csv",
    index=False,
)

pd.concat(
    secondary_residual,
    ignore_index=True,
).to_csv(
    OUTDIR / "ae6_secondary_joint_vs_devonly_MLM_residual_structure.csv",
    index=False,
)

print("PRIMARY joint-representation projection medians:")
display(primary_wide.round(4))

print("PRIMARY population-rank concordance:")
display(primary_rank_corr.round(3))

print("PRIMARY per-cell Burdziak concordance:")
display(primary_cell_corr.round(3))

print("PRIMARY descending projection order:")
display(primary_rank_order)

print("SECONDARY MLM training-composition comparison:")
display(
    secondary_projection[
        secondary_projection["population"].isin(ORDER_POPULATIONS)
    ]
    .pivot_table(
        index="population",
        columns="model",
        values="projection_median",
        aggfunc="first",
    )
    .reindex(ORDER_POPULATIONS)
    .round(4)
)


PRIMARY joint-representation projection medians:


representation,joint_MLM_adultCANCER,joint_MLM_adultDEV,joint_expression_PCA,joint_scVI_seed17,joint_scVI_seed42
population,,,,,
adult_ductal,0.7445,0.7488,0.4496,0.9997,0.9997
N1,0.5961,0.5961,0.3367,0.9997,0.9997
N2,0.5269,0.5269,0.3469,0.9997,0.9997
K1,0.5247,0.5247,0.3319,0.9996,0.9996
K1.5,0.5257,0.5257,0.3265,0.9996,0.9996
K2,0.5317,0.5317,0.3855,0.9997,0.9996
K3,0.5306,0.5306,0.3325,0.9997,0.9996
K4,0.5576,0.5576,0.3054,0.9997,0.9996
K5,0.5640,0.5640,0.3971,0.9996,0.9995


PRIMARY population-rank concordance:


representation,joint_MLM_adultCANCER,joint_MLM_adultDEV,joint_expression_PCA,joint_scVI_seed17,joint_scVI_seed42
representation,,,,,
joint_MLM_adultCANCER,1.000,1.000,0.539,0.236,0.333
joint_MLM_adultDEV,1.000,1.000,0.539,0.236,0.333
joint_expression_PCA,0.539,0.539,1.000,-0.067,-0.115
joint_scVI_seed17,0.236,0.236,-0.067,1.000,0.806
joint_scVI_seed42,0.333,0.333,-0.115,0.806,1.000


PRIMARY per-cell Burdziak concordance:


,joint_expression_PCA,joint_scVI_seed17,joint_scVI_seed42,joint_MLM
joint_expression_PCA,1.000,0.087,-0.154,0.051
joint_scVI_seed17,0.087,1.000,0.490,0.207
joint_scVI_seed42,-0.154,0.490,1.000,0.180
joint_MLM,0.051,0.207,0.180,1.000


PRIMARY descending projection order:


,representation,descending_projection_order
0,joint_MLM_adultCANCER,adult_ductal > N1 > K5 > K4 > K6 > K2 > K3 > N...
1,joint_MLM_adultDEV,adult_ductal > N1 > K5 > K4 > K6 > K2 > K3 > N...
2,joint_expression_PCA,adult_ductal > K5 > K2 > K6 > N2 > N1 > K3 > K...
3,joint_scVI_seed17,N2 > N1 > K4 > K2 > adult_ductal > K3 > K1.5 >...
4,joint_scVI_seed42,N1 > adult_ductal > N2 > K4 > K3 > K1.5 > K2 >...


SECONDARY MLM training-composition comparison:


model,development_only_MLM_seed17,development_only_MLM_seed42,joint_MLM_adultCANCER,joint_MLM_adultDEV
population,,,,
adult_ductal,0.9485,0.9607,0.7445,0.7488
N1,0.9677,0.9625,0.5961,0.5961
N2,0.9541,0.9569,0.5269,0.5269
K1,0.9492,0.9584,0.5247,0.5247
K1.5,0.9489,0.9593,0.5257,0.5257
K2,0.9481,0.9567,0.5317,0.5317
K3,0.9494,0.9601,0.5306,0.5306
K4,0.9495,0.9607,0.5576,0.5576
K5,0.9481,0.9561,0.5640,0.5640


## 10. Interpretation and handoff

The intended decision logic is:

1. **Absolute projection fractions are representation-dependent.** Do not compare a raw scVI projection fraction numerically with an MLM or expression-PCA fraction without matching developmental variance.
2. **Directional robustness is the key cross-representation biological test.** The principal comparison is the sign and approximate magnitude of the equal-day developmental-direction cosines computed from equal-biological-sample-weighted cancer centroids.
3. **Residual structure should be interpreted qualitatively across representations.** Low-dimensional/state-associated residual organization can be robust even when the fraction concentrated in particular residual PCs is representation-specific.
4. **Adult-normal calibration remains anchored in AE5.** The matched-variance AE6 sensitivity below deliberately omits adult cells so it does not overstate adult-vs-malignant reconstruction robustness across all representations.


In [ ]:
print("AE6 outputs written to:", OUTDIR)

print("\nPRIMARY files to inspect first:")
for name in [
    "ae6_primary_joint_projection_ordering.csv",
    "ae6_primary_joint_population_rank_spearman.csv",
    "ae6_primary_joint_burdziak_cell_projection_spearman.csv",
    "ae6_primary_joint_revised_ae3_direction_cosines.csv",
    "ae6_primary_joint_kseries_residual_structure.csv",
]:
    p = OUTDIR / name
    print(("OK   " if p.exists() else "PEND "), name)

print("\nSECONDARY training-composition controls:")
for name in [
    "ae6_secondary_joint_vs_devonly_MLM_projection.csv",
    "ae6_secondary_joint_vs_devonly_MLM_direction_cosines.csv",
    "ae6_secondary_joint_vs_devonly_MLM_residual_structure.csv",
]:
    p = OUTDIR / name
    print(("OK   " if p.exists() else "PEND "), name)


WP6 outputs written to: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation_dependence_joint_primary

PRIMARY files to inspect first:
OK    wp6_primary_joint_projection_ordering.csv
OK    wp6_primary_joint_population_rank_spearman.csv
OK    wp6_primary_joint_burdziak_cell_projection_spearman.csv
OK    wp6_primary_joint_revised_wp3_direction_cosines.csv
OK    wp6_primary_joint_kseries_residual_structure.csv

SECONDARY training-composition controls:
OK    wp6_secondary_joint_vs_devonly_MLM_projection.csv
OK    wp6_secondary_joint_vs_devonly_MLM_direction_cosines.csv
OK    wp6_secondary_joint_vs_devonly_MLM_residual_structure.csv


### First-pass outputs

Before the matched-variance sensitivity, inspect:

1. `wp6_primary_joint_revised_wp3_direction_cosines.csv`
2. `wp6_primary_joint_kseries_residual_structure.csv`
3. `wp6_secondary_joint_vs_devonly_MLM_direction_cosines.csv`
4. `wp6_secondary_joint_vs_devonly_MLM_residual_structure.csv`

The matched-variance tables generated later are the decisive cross-representation projection/direction sensitivity outputs.


In [ ]:
files_to_show = {
    "PRIMARY revised AE3 direction cosines":
        OUTDIR / "ae6_primary_joint_revised_ae3_direction_cosines.csv",

    "PRIMARY K-series residual structure":
        OUTDIR / "ae6_primary_joint_kseries_residual_structure.csv",

    "SECONDARY joint vs dev-only MLM direction cosines":
        OUTDIR / "ae6_secondary_joint_vs_devonly_MLM_direction_cosines.csv",

    "SECONDARY joint vs dev-only MLM residual structure":
        OUTDIR / "ae6_secondary_joint_vs_devonly_MLM_residual_structure.csv",
}

for title, path in files_to_show.items():
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    display(pd.read_csv(path))


PRIMARY revised WP3 direction cosines


,centroid_weighting,cancer_centroid_weighting,transition,developmental_direction,cosine,abs_cosine,representation
0,cell,equal_biological_sample,early,MPC_to_tip_acinar,-0.525795,0.525795,joint_expression_PCA
1,cell,equal_biological_sample,early,MPC_to_endocrine_islet,0.079262,0.079262,joint_expression_PCA
2,cell,equal_biological_sample,early,endocrine_maturation,-0.213454,0.213454,joint_expression_PCA
3,cell,equal_biological_sample,early,biliary_maturation,0.165536,0.165536,joint_expression_PCA
4,cell,equal_biological_sample,late,MPC_to_tip_acinar,-0.590126,0.590126,joint_expression_PCA
...,...,...,...,...,...,...,...
91,equal_day,equal_biological_sample,late,biliary_maturation,-0.083133,0.083133,joint_MLM
92,equal_day,equal_biological_sample,total,MPC_to_tip_acinar,-0.386772,0.386772,joint_MLM
93,equal_day,equal_biological_sample,total,MPC_to_endocrine_islet,0.049289,0.049289,joint_MLM
94,equal_day,equal_biological_sample,total,endocrine_maturation,-0.181792,0.181792,joint_MLM



PRIMARY K-series residual structure


,n_K_cells,residual_pc_variance_first10,residual_pc_variance_first30,between_condition_fraction_raw_residual,between_broad_state_fraction_raw_residual,max_eta2_condition_first10,max_eta2_broad_state_first10,mean_eta2_condition_first10,mean_eta2_broad_state_first10,representation
0,22505,0.829583,0.934665,0.277098,0.184368,0.553002,0.525407,0.220726,0.085049,joint_expression_PCA
1,22505,0.529143,0.776569,0.125995,0.089807,0.659851,0.634632,0.128386,0.077301,joint_scVI_seed17
2,22505,0.539222,0.785875,0.079011,0.038989,0.309123,0.290241,0.085912,0.032789,joint_scVI_seed42
3,22505,0.667246,0.879329,0.187723,0.086395,0.553034,0.377748,0.206802,0.078837,joint_MLM



SECONDARY joint vs dev-only MLM direction cosines


,centroid_weighting,cancer_centroid_weighting,transition,developmental_direction,cosine,abs_cosine,model
0,cell,equal_biological_sample,early,MPC_to_tip_acinar,-0.334365,0.334365,joint_MLM
1,cell,equal_biological_sample,early,MPC_to_endocrine_islet,0.048137,0.048137,joint_MLM
2,cell,equal_biological_sample,early,endocrine_maturation,-0.185958,0.185958,joint_MLM
3,cell,equal_biological_sample,early,biliary_maturation,0.247631,0.247631,joint_MLM
4,cell,equal_biological_sample,late,MPC_to_tip_acinar,-0.131045,0.131045,joint_MLM
...,...,...,...,...,...,...,...
67,equal_day,equal_biological_sample,late,biliary_maturation,-0.012381,0.012381,development_only_MLM_seed42
68,equal_day,equal_biological_sample,total,MPC_to_tip_acinar,-0.565668,0.565668,development_only_MLM_seed42
69,equal_day,equal_biological_sample,total,MPC_to_endocrine_islet,-0.199795,0.199795,development_only_MLM_seed42
70,equal_day,equal_biological_sample,total,endocrine_maturation,-0.223462,0.223462,development_only_MLM_seed42



SECONDARY joint vs dev-only MLM residual structure


,n_K_cells,residual_pc_variance_first10,residual_pc_variance_first30,between_condition_fraction_raw_residual,between_broad_state_fraction_raw_residual,max_eta2_condition_first10,max_eta2_broad_state_first10,mean_eta2_condition_first10,mean_eta2_broad_state_first10,model
0,22505,0.667246,0.879329,0.187723,0.086395,0.553034,0.377748,0.206802,0.078837,joint_MLM
1,22505,0.497672,0.734572,0.026699,0.014816,0.143778,0.098576,0.041710,0.020892,development_only_MLM_seed17
2,22505,0.517231,0.744000,0.033318,0.020609,0.082559,0.067441,0.045839,0.027677,development_only_MLM_seed42


For the primary direction table, pivot the **equal-day developmental directions**. Cancer centroids in every row are already equal-biological-sample weighted.


In [ ]:
cos = pd.read_csv(OUTDIR / "ae6_primary_joint_revised_ae3_direction_cosines.csv")

display(
    cos[
        cos["centroid_weighting"] == "equal_day"
    ].pivot_table(
        index=[
            "transition",
            "developmental_direction",
        ],
        columns="representation",
        values="cosine",
    ).round(3)
)

representation                      joint_MLM  joint_expression_PCA  \
transition developmental_direction                                    
early      MPC_to_endocrine_islet       0.063                 0.091   
           MPC_to_tip_acinar           -0.319                -0.465   
           biliary_maturation           0.247                 0.137   
           endocrine_maturation        -0.196                -0.222   
late       MPC_to_endocrine_islet      -0.017                -0.089   
           MPC_to_tip_acinar           -0.142                -0.551   
           biliary_maturation          -0.083                -0.035   
           endocrine_maturation         0.006                 0.002   
total      MPC_to_endocrine_islet       0.049                 0.041   
           MPC_to_tip_acinar           -0.387                -0.509   
           biliary_maturation           0.183                 0.091   
           endocrine_maturation        -0.182                -0.164   

representation                      joint_scVI_seed17  joint_scVI_seed42  
transition developmental_direction                                        
early      MPC_to_endocrine_islet              -0.087              0.055  
           MPC_to_tip_acinar                   -0.265             -0.162  
           biliary_maturation                  -0.013              0.051  
           endocrine_maturation                 0.132              0.070  
late       MPC_to_endocrine_islet               0.078             -0.087  
           MPC_to_tip_acinar                    0.051             -0.038  
           biliary_maturation                   0.132             -0.089  
           endocrine_maturation                 0.057             -0.003  
total      MPC_to_endocrine_islet              -0.012             -0.017  
           MPC_to_tip_acinar                   -0.166             -0.152  
           biliary_maturation                   0.081             -0.021  
           endocrine_maturation                 0.140              0.053

the residual table

In [ ]:
res = pd.read_csv(OUTDIR / "ae6_primary_joint_kseries_residual_structure.csv") 
display(res.set_index("representation").round(4))

,n_K_cells,residual_pc_variance_first10,residual_pc_variance_first30,between_condition_fraction_raw_residual,between_broad_state_fraction_raw_residual,max_eta2_condition_first10,max_eta2_broad_state_first10,mean_eta2_condition_first10,mean_eta2_broad_state_first10
representation,,,,,,,,,
joint_expression_PCA,22505,0.8296,0.9347,0.2771,0.1844,0.5530,0.5254,0.2207,0.0850
joint_scVI_seed17,22505,0.5291,0.7766,0.1260,0.0898,0.6599,0.6346,0.1284,0.0773
joint_scVI_seed42,22505,0.5392,0.7859,0.0790,0.0390,0.3091,0.2902,0.0859,0.0328
joint_MLM,22505,0.6672,0.8793,0.1877,0.0864,0.5530,0.3777,0.2068,0.0788


Does the key directional conclusion survive leave-tumor-out MLM learning? The secondary table answers this using the same final AE3 equal-sample malignant centroids.


In [ ]:
sec_cos = pd.read_csv(OUTDIR /"ae6_secondary_joint_vs_devonly_MLM_direction_cosines.csv")

display(
    sec_cos[
        sec_cos["centroid_weighting"] == "equal_day"
    ].pivot_table(
        index=[
            "transition",
            "developmental_direction",
        ],
        columns="model",
        values="cosine",
    ).round(3)
)

model                               development_only_MLM_seed17  \
transition developmental_direction                                
early      MPC_to_endocrine_islet                        -0.167   
           MPC_to_tip_acinar                             -0.471   
           biliary_maturation                             0.199   
           endocrine_maturation                          -0.283   
late       MPC_to_endocrine_islet                        -0.459   
           MPC_to_tip_acinar                             -0.318   
           biliary_maturation                            -0.143   
           endocrine_maturation                          -0.090   
total      MPC_to_endocrine_islet                        -0.222   
           MPC_to_tip_acinar                             -0.461   
           biliary_maturation                             0.148   
           endocrine_maturation                          -0.260   

model                               development_only_MLM_seed42  joint_MLM  
transition developmental_direction                                          
early      MPC_to_endocrine_islet                        -0.147      0.063  
           MPC_to_tip_acinar                             -0.563     -0.319  
           biliary_maturation                             0.346      0.247  
           endocrine_maturation                          -0.233     -0.196  
late       MPC_to_endocrine_islet                        -0.420     -0.017  
           MPC_to_tip_acinar                             -0.445     -0.142  
           biliary_maturation                            -0.012     -0.083  
           endocrine_maturation                          -0.123      0.006  
total      MPC_to_endocrine_islet                        -0.200      0.049  
           MPC_to_tip_acinar                             -0.566     -0.387  
           biliary_maturation                             0.298      0.183  
           endocrine_maturation                          -0.223     -0.182

## 11. Matched-developmental-variance sensitivity

This is the main cross-representation calibration.

`K=75` is meaningful inside the MLM analysis but corresponds to very different cumulative developmental variance in expression PCA and scVI. We therefore repeat the central DEV/Burdziak analyses at:

- 90% developmental variance;
- 95% developmental variance;
- the representation-specific `K` matching the amount of developmental variance captured by MLM `K=75`.

Adult cells are deliberately omitted from this sensitivity. AE5 remains the independent adult-normal calibration.


*Recover the four primary joint DEV/Burdziak representations without retraining scVI or recomputing MLM embeddings.*


In [ ]:
# ============================================================
# AE6 closure: matched developmental-variance sensitivity
# ============================================================

# Expression-PCA coordinates already exist from this audited run.
matched_var_reps = {
    "joint_expression_PCA": (
        Zdev_expr,
        Zcan_expr,
    ),
}


# ------------------------------------------------------------
# Reload the two SAVED scVI models from this audited AE6 run.
# No retraining.
# ------------------------------------------------------------

for seed in SCVI_SEEDS:
    print(
        "Reloading joint scVI seed",
        seed,
    )

    joint_tmp = scvi_joint.copy()

    scvi.model.SCVI.setup_anndata(
        joint_tmp
    )

    model_tmp = scvi.model.SCVI.load(
        OUTDIR / f"joint_scVI_seed{seed}_model",
        adata=joint_tmp,
    )

    Zjoint_tmp = (
        model_tmp
        .get_latent_representation(
            give_mean=True
        )
        .astype(np.float32)
    )

    matched_var_reps[
        f"joint_scVI_seed{seed}"
    ] = (
        Zjoint_tmp[:A_dev.n_obs],
        Zjoint_tmp[A_dev.n_obs:],
    )

    del (
        model_tmp,
        joint_tmp,
        Zjoint_tmp,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()


# ------------------------------------------------------------
# Reload the definitive audited primary joint MLM cache.
# ------------------------------------------------------------

print(
    "Reloading audited joint MLM embeddings"
)

(
    Zdev_joint_mlm,
    Zcan_joint_mlm,
) = load_audited_mlm_embeddings(
    JOINT_MODEL_DIR
)

matched_var_reps[
    "joint_MLM"
] = (
    Zdev_joint_mlm,
    Zcan_joint_mlm,
)

print(
    "\nRecovered representations:"
)

for name, (Zd, Zc) in matched_var_reps.items():
    print(
        name,
        "DEV",
        Zd.shape,
        "CANCER",
        Zc.shape,
    )


Reloading joint scVI seed 17
INFO     File                                                                                                      
         /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation
         _dependence_joint_primary/joint_scVI_seed17_model/model.pt already downloaded                             
Reloading joint scVI seed 42
INFO     File                                                                                                      
         /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation
         _dependence_joint_primary/joint_scVI_seed42_model/model.pt already downloaded                             
Reloading audited joint MLM embeddings
Audited MLM cache: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/models/joint/seed_17/model | DEV (34760, 256) | CANCER (28131, 256)

Recovered representations:
joint_express

*Determine the $K$ required for matched developmental variance*

In [47]:
def developmental_variance_curve(
    Z_dev,
    ref_idx,
):
    """
    PCA spectrum of the developmental reference cells
    in one representation.
    """
    Z_dev = np.asarray(
        Z_dev,
        dtype=np.float32,
    )

    mu = Z_dev[
        ref_idx
    ].mean(axis=0)

    Xref = (
        Z_dev[ref_idx]
        - mu
    )

    max_rank = min(
        Xref.shape[1],
        Xref.shape[0] - 1,
    )

    pca = PCA(
        n_components=max_rank,
        svd_solver="full",
    )

    pca.fit(Xref)

    cumulative = np.cumsum(
        pca.explained_variance_ratio_
    )

    return cumulative


def k_for_variance(
    cumulative,
    target,
):
    """
    Smallest K reaching the requested cumulative
    developmental variance.
    """
    k = (
        np.searchsorted(
            cumulative,
            target,
            side="left",
        )
        + 1
    )

    return min(
        int(k),
        len(cumulative),
    )

*First calculate the cumulative developmental variance captured by audited joint-MLM `K=75`.*


In [48]:
cumvar = {}

for name, (Zdev, _) in matched_var_reps.items():
    cumvar[name] = developmental_variance_curve(
        Zdev,
        ref_idx,
    )


mlm_K75_variance = float(cumvar["joint_MLM"][74])
print("Joint MLM developmental variance at K=75:",f"{mlm_K75_variance:.4f}")

Joint MLM developmental variance at K=75: 0.9800


*Define the matched developmental-variance targets.*


In [49]:
variance_targets = {
    "90pct": 0.90,
    "95pct": 0.95,
    "MLM_K75_equivalent":
        mlm_K75_variance,
}


*Construct the representation-specific `K` table.*


In [ ]:
k_rows = []

for name, cumulative in cumvar.items():

    for target_name, target in variance_targets.items():

        k = k_for_variance(
            cumulative,
            target,
        )

        achieved = float(
            cumulative[k - 1]
        )

        k_rows.append({
            "representation": name,
            "target": target_name,
            "target_variance": target,
            "K": k,
            "achieved_variance": achieved,
        })


matched_K_table = pd.DataFrame(k_rows)

display(matched_K_table.pivot(index="representation",columns="target",values="K"))

matched_K_table.to_csv(OUTDIR / "ae6_matched_variance_K_table.csv",index=False)

target,90pct,95pct,MLM_K75_equivalent
representation,,,
joint_MLM,24,39,75
joint_expression_PCA,167,203,228
joint_scVI_seed17,29,33,36
joint_scVI_seed42,29,33,36


*Recalculate the three important AE6 quantities at each matched-variance target: projection, final-AE3 direction cosines, and K-series residual structure.*


In [ ]:
matched_variance_results = {}

projection_tables = []
direction_tables = []
residual_tables = []


for rep_name, (Zdev, Zcan) in matched_var_reps.items():

    cumulative = cumvar[
        rep_name
    ]

    for target_name, target in variance_targets.items():

        k = k_for_variance(
            cumulative,
            target,
        )

        tag = (
            f"{rep_name}__{target_name}"
        )

        print(
            "\n",
            tag,
            "| K =", k,
        )

        # Adult deliberately omitted here:
        # this is the common DEV/Burdziak sensitivity.
        res = evaluate_representation(
            tag,
            Zdev,
            Zcan,
            Z_adult=None,
            k=k,
        )

        matched_variance_results[
            tag
        ] = res


        # Projection
        t = (
            res[
                "projection_summary"
            ]
            .copy()
        )

        t["representation"] = rep_name
        t["variance_target"] = target_name
        t["K"] = k

        projection_tables.append(
            t
        )


        # Revised AE3 directions
        t = (
            res[
                "direction_cosines"
            ]
            .copy()
        )

        t["representation"] = rep_name
        t["variance_target"] = target_name
        t["K"] = k

        direction_tables.append(
            t
        )


        # K-series residual structure
        t = (
            res[
                "residual_summary"
            ]
            .copy()
        )

        t["representation"] = rep_name
        t["variance_target"] = target_name
        t["K"] = k

        residual_tables.append(
            t
        )


matched_projection = pd.concat(
    projection_tables,
    ignore_index=True,
)

matched_directions = pd.concat(
    direction_tables,
    ignore_index=True,
)

matched_residual = pd.concat(
    residual_tables,
    ignore_index=True,
)


matched_projection.to_csv(
    OUTDIR /
    "ae6_matched_variance_projection.csv",
    index=False,
)

matched_directions.to_csv(
    OUTDIR /
    "ae6_matched_variance_direction_cosines.csv",
    index=False,
)

matched_residual.to_csv(
    OUTDIR /
    "ae6_matched_variance_residual_structure.csv",
    index=False,
)


 joint_expression_PCA__90pct | K = 167

 joint_expression_PCA__95pct | K = 203

 joint_expression_PCA__MLM_K75_equivalent | K = 228

 joint_scVI_seed17__90pct | K = 29

 joint_scVI_seed17__95pct | K = 33

 joint_scVI_seed17__MLM_K75_equivalent | K = 36

 joint_scVI_seed42__90pct | K = 29

 joint_scVI_seed42__95pct | K = 33

 joint_scVI_seed42__MLM_K75_equivalent | K = 36

 joint_MLM__90pct | K = 24

 joint_MLM__95pct | K = 39

 joint_MLM__MLM_K75_equivalent | K = 75


*Print the matched-variance projection comparison. Held-out development, N1/N2 and the K-series are shown; adult is intentionally not included in this common sensitivity.*


In [52]:
display(
    matched_projection[
        matched_projection[
            "population"
        ].isin([
            "heldout_dev_ductal",
            "N1",
            "N2",
            "K_series_all",
        ])
    ]
    .pivot_table(
        index=[
            "variance_target",
            "population",
        ],
        columns="representation",
        values="projection_median",
    )
    .round(4)
)

representation                         joint_MLM  joint_expression_PCA  \
variance_target    population                                            
90pct              K_series_all           0.1593                0.3861   
                   N1                     0.1866                0.3531   
                   N2                     0.1441                0.3673   
                   heldout_dev_ductal     0.8893                0.8613   
95pct              K_series_all           0.3130                0.4011   
                   N1                     0.3200                0.3615   
                   N2                     0.2997                0.3796   
                   heldout_dev_ductal     0.9459                0.9301   
MLM_K75_equivalent K_series_all           0.5410                0.4105   
                   N1                     0.5961                0.3699   
                   N2                     0.5269                0.3898   
                   heldout_dev_ductal     0.9782                0.9740   

representation                         joint_scVI_seed17  joint_scVI_seed42  
variance_target    population                                                
90pct              K_series_all                   0.5289             0.5006  
                   N1                             0.4038             0.2939  
                   N2                             0.4844             0.5008  
                   heldout_dev_ductal             0.9080             0.9052  
95pct              K_series_all                   0.6465             0.6186  
                   N1                             0.5246             0.3449  
                   N2                             0.5508             0.5704  
                   heldout_dev_ductal             0.9582             0.9572  
MLM_K75_equivalent K_series_all                   0.7425             0.7355  
                   N1                             0.5738             0.3927  
                   N2                             0.6372             0.6612  
                   heldout_dev_ductal             0.9866             0.9878

*For the directional result, concentrate on the most portable WP3 result: early and total displacement relative to MPC→tip/acinar. Cancer state centroids are equal-biological-sample weighted.*


In [53]:
display(
    matched_directions[
        (
            matched_directions[
                "centroid_weighting"
            ] == "equal_day"
        )
        &
        (
            matched_directions[
                "transition"
            ].isin([
                "early",
                "total",
            ])
        )
        &
        (
            matched_directions[
                "developmental_direction"
            ] == "MPC_to_tip_acinar"
        )
    ]
    .pivot_table(
        index=[
            "variance_target",
            "transition",
        ],
        columns="representation",
        values="cosine",
    )
    .round(3)
)

representation                 joint_MLM  joint_expression_PCA  \
variance_target    transition                                    
90pct              early          -0.513                -0.414   
                   total          -0.591                -0.456   
95pct              early          -0.413                -0.403   
                   total          -0.489                -0.443   
MLM_K75_equivalent early          -0.319                -0.389   
                   total          -0.387                -0.430   

representation                 joint_scVI_seed17  joint_scVI_seed42  
variance_target    transition                                        
90pct              early                  -0.272             -0.245  
                   total                  -0.221             -0.504  
95pct              early                  -0.346             -0.195  
                   total                  -0.217             -0.363  
MLM_K75_equivalent early                  -0.322             -0.226  
                   total                  -0.176             -0.195

*Finally, compare development-unexplained residual structure across matched developmental variance.*


In [54]:
display(
    matched_residual[
        [
            "variance_target",
            "representation",
            "K",
            "residual_pc_variance_first10",
            "residual_pc_variance_first30",
            "between_condition_fraction_raw_residual",
            "between_broad_state_fraction_raw_residual",
            "max_eta2_condition_first10",
            "max_eta2_broad_state_first10",
        ]
    ]
    .sort_values([
        "variance_target",
        "representation",
    ])
    .round(4)
)

,variance_target,representation,K,residual_pc_variance_first10,residual_pc_variance_first30,between_condition_fraction_raw_residual,between_broad_state_fraction_raw_residual,max_eta2_condition_first10,max_eta2_broad_state_first10
9,90pct,joint_MLM,24,0.6634,0.8682,0.1817,0.0795,0.5136,0.3360
0,90pct,joint_expression_PCA,167,0.8712,0.9753,0.2910,0.1941,0.5562,0.5287
3,90pct,joint_scVI_seed17,29,0.9985,0.9995,0.2648,0.1531,0.6321,0.5325
6,90pct,joint_scVI_seed42,29,0.9986,0.9996,0.3032,0.1666,0.6368,0.5399
10,95pct,joint_MLM,39,0.6675,0.8732,0.1858,0.0831,0.5134,0.3778
1,95pct,joint_expression_PCA,203,0.8880,0.9901,0.2964,0.1982,0.5561,0.5288
4,95pct,joint_scVI_seed17,33,0.9978,0.9992,0.3234,0.2308,0.6761,0.6501
7,95pct,joint_scVI_seed42,33,0.9981,0.9993,0.3463,0.2830,0.6415,0.5947
11,MLM_K75_equivalent,joint_MLM,75,0.6672,0.8793,0.1877,0.0864,0.5530,0.3777
2,MLM_K75_equivalent,joint_expression_PCA,228,0.9025,1.0000,0.3016,0.2018,0.5600,0.5324


## 12. Final audit manifest and output inventory

This final cell records the exact audited inputs, split, biological definitions, representation settings, and expected WP6 outputs. Run it only after all analyses above complete successfully.

In [ ]:
expected_output_files = [
    "joint_expression_pca_hvgs.csv",
    "joint_expression_pca_model.npz",
    "ae6_primary_joint_projection_ordering.csv",
    "ae6_primary_joint_population_rank_spearman.csv",
    "ae6_primary_joint_population_rank_order.csv",
    "ae6_primary_joint_burdziak_cell_projection_spearman.csv",
    "ae6_primary_joint_burdziak_cell_projection_scores.csv.gz",
    "ae6_primary_joint_revised_ae3_direction_cosines.csv",
    "ae6_primary_joint_kseries_residual_structure.csv",
    "ae6_secondary_joint_vs_devonly_MLM_projection.csv",
    "ae6_secondary_joint_vs_devonly_MLM_direction_cosines.csv",
    "ae6_secondary_joint_vs_devonly_MLM_residual_structure.csv",
    "ae6_matched_variance_K_table.csv",
    "ae6_matched_variance_projection.csv",
    "ae6_matched_variance_direction_cosines.csv",
    "ae6_matched_variance_residual_structure.csv",
]

audit_payload = {
    "ae6_version":
        "audited_final_v1",

    "ae1b_manifest":
        str(MANIFEST_PATH),

    "audited_dev_h5ad":
        str(DEV_H5AD),

    "audited_cancer_h5ad":
        str(CANCER_H5AD),

    "primary_joint_mlm_seed":
        int(PRIMARY_SEED),

    "primary_joint_mlm_model_dir":
        str(JOINT_MODEL_DIR),

    "development_only_model_dirs":
        {
            str(k): str(v)
            for k, v in DEV_ONLY_MODEL_DIRS.items()
        },

    "ae5_cell_metrics":
        str(AE5_CELL_METRICS),

    "developmental_split": {
        "seed":
            int(RANDOM_SEED),
        "identity_column":
            str(DEV_LABEL_COL),
        "stage_column":
            (
                None
                if DEV_SPLIT_DAY_COL is None
                else str(DEV_SPLIT_DAY_COL)
            ),
        "reference_fraction":
            float(DEV_REFERENCE_FRACTION),
        "n_reference":
            int(len(ref_idx)),
        "n_heldout":
            int(len(hold_idx)),
    },

    "ae3_developmental_state_counts":
        {
            k: int(v)
            for k, v in observed_ae3_counts.items()
        },

    "burdziak_biological_samples_per_state":
        {
            k: int(v)
            for k, v in sample_counts_by_state.items()
        },

    "cancer_centroid_weighting":
        "equal_biological_sample_by_batch",

    "expression_pca": {
        "n_hvg":
            int(N_HVG),
        "representation_dim":
            int(PCA_REP_DIM),
    },

    "scvi": {
        "version":
            str(scvi.__version__),
        "latent_dim":
            int(SCVI_LATENT_DIM),
        "seeds":
            [int(x) for x in SCVI_SEEDS],
        "max_epochs":
            int(SCVI_MAX_EPOCHS),
        "gene_likelihood":
            "nb",
        "dispersion":
            "gene",
        "batch_key":
            None,
        "adult_query_training":
            False,
        "adult_latent_summary":
            "posterior_mean",
    },

    "fixed_K":
        int(K_DEV),

    "output_directory":
        str(OUTDIR),

    "expected_output_files":
        expected_output_files,

    "scvi_model_directories": [
        str(
            OUTDIR
            / f"joint_scVI_seed{seed}_model"
        )
        for seed in SCVI_SEEDS
    ],
}

AUDIT_OUT = (
    OUTDIR
    / "ae6_audit_manifest.json"
)

with open(
    AUDIT_OUT,
    "w",
) as f:
    json.dump(
        audit_payload,
        f,
        indent=2,
    )

print("AE6 audit manifest:",AUDIT_OUT)
print("\nExpected tabular/model outputs:")

missing_outputs = []

for name in expected_output_files:
    p = OUTDIR / name
    ok = p.exists()
    print(
        ("OK   " if ok else "MISS "),
        p,
    )
    if not ok:
        missing_outputs.append(
            str(p)
        )

for seed in SCVI_SEEDS:
    p = (
        OUTDIR
        / f"joint_scVI_seed{seed}_model"
    )
    ok = p.exists()
    print(
        ("OK   " if ok else "MISS "),
        p,
    )
    if not ok:
        missing_outputs.append(
            str(p)
        )

if missing_outputs:
    raise FileNotFoundError(
        "AE6 finished with missing expected outputs:\n"
        + "\n".join(missing_outputs)
    )

print("\nAE6 AUDIT STATUS: PASS")
print("All outputs are under:",OUTDIR)

WP6 audit manifest: /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation_dependence_joint_primary/wp6_audit_manifest.json

Expected tabular/model outputs:
OK    /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation_dependence_joint_primary/joint_expression_pca_hvgs.csv
OK    /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation_dependence_joint_primary/joint_expression_pca_model.npz
OK    /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation_dependence_joint_primary/wp6_primary_joint_projection_ordering.csv
OK    /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/analysis/wp6_representation_dependence_joint_primary/wp6_primary_joint_population_rank_spearman.csv
OK    /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/revision_audited_mlm/anal